# H&M GraphSAGE · M1~M5_D 8모형 · seed49

**GraphSAGE 기반에서 확정 M2·M3·M4의 독립 효과와 결합 효과를 비교**합니다. 모두 같은 GraphSAGE M1을 기준으로 비교하며 기존 LightGCN/NGCF/GAT 성능·체크포인트는 GraphSAGE 결과로 재사용하지 않습니다.

|순서|모형|M2 표현|M3 그래프|M4 손실|
|---|---|---|---|---|
|1|M1|없음|이진 연결|무가중|
|2|M2|적응형 N/V|이진 연결|무가중|
|3|M3|없음|중심화 N/V|무가중|
|4|M4|없음|이진 연결|고객 내 평균1|
|5|M5_A|적응형 N/V|중심화 N/V|무가중|
|6|M5_B|적응형 N/V|이진 연결|고객 내 평균1|
|7|M5_C|없음|중심화 N/V|고객 내 평균1|
|8|M5_D|적응형 N/V|중심화 N/V|고객 내 평균1|

## 고정 실험조건
- L4 GPU, seed49, train 2018-09-20~2020-09-15 / test 2020-09-16~22. 신규 상품 추천: 학습 구매쌍을 test 정답·후보에서 제외. MIN_ITEM_INTER=1, uniform 음성1개.
- 모형당 **300epoch 마지막 체크포인트만 test 1회 평가**. validation·holdout·조기종료·중간test·추가시드 자동실행 없음. 결과를 보고 계수·epoch를 바꾸지 않습니다.
- ID64·2층64·batch131072·Adam lr.0005·ID L2 .001·M2 공유 L2 1e-6·GraphSAGE 변환/편향 L2 1e-6. 하나의 optimizer로 ID·GraphSAGE·M2를 공동학습합니다. 사전학습 동결·외부재정렬 없음.
- M2 N/V·아이템 경제 속성·관측량 보조계수·RBF·양성상품의 현재 구매자 제외·학습형 축계수/rho·직교결합은 유지하며 출력64차원입니다.
- M4 고객 내부 구매행 평균1 가중치를 그대로 사용합니다. CLV는 학습구간 최근365일의 historical CLV proxy이며 미래 CLV/증분매출이 아닙니다.

## 이번 GraphSAGE의 정의와 M3 적용
이웃의 평균 표현과 자기 표현을 따로 결합하는 **mean GraphSAGE**입니다. 기존 프로젝트의 GraphSAGE 추천용 설정(ELU·층별 L2 정규화·0/1/2층 평균)을 사용합니다. 원논문의 데이터/분류기 전체 재현은 아닙니다.

`P[u,v] = A_w[u,v] / sum_v A_w[u,v]`

`H_next = L2normalize(ELU(concat(H, P @ H) @ W + b))`

`Z = (E + H1 + H2) / 3; score(u,i) = inner_product(Z_u,Z_i)`

- 각 층은 자기64차원과 이웃64차원을 합친128차원을64차원으로 변환합니다. 두 층을 사용하고 dropout은0입니다. M2가 있으면 이 Z에 확정된 직교 경제표현 결합을 적용한 뒤 추천점수를 계산합니다.
- 이웃 표본추출 없이 전체 이웃을 사용합니다. 자기 표현은 concat으로 들어가므로 인접행렬에 자기연결을 추가하지 않습니다. 이웃 없는 노드의 이웃평균은0이며 자기 경로는 남습니다.
- M3가 없으면 관측 엣지 가중치1의 평균, M3가 있으면 **기존 중심화 N/V 원시 가중치로 계산한 가중평균**입니다. 이미 대칭 정규화된 LightGCN 행렬을 사용하지 않습니다.
- M3 입력·계수는 그대로지만 행별 가중치 합으로 다시 나누기 때문에 공통 행 배율은 상쇄됩니다. 원시 M3에서 점검한 양쪽 총량 성질이 최종 GraphSAGE 전파에서도 그대로 유지된다고 주장하지 않습니다.
- 학습형 ID를 쓰는 추천 실험으로, 학습하지 않은 새 사용자·상품에 대한 귀납 일반화 검증은 아닙니다. 실제 전파/층결합 차이를 protocol에 기록하고 백본별 M1 대비 효과를 비교합니다.

## 입력 복구·실행 최적화 (2026-10-23)
- 학교 Drive의 거래 Parquet를 개인 Drive 원본과 동일한 바이트로 복구했습니다. 기존 5조각 ZIP은 잘못 변환된 입력이므로 사용하지 않습니다.
- 원본 두 파일의 통합 SHA256은 기존 M2 기록과 일치하며, 아래 준비 셀은 참조 ZIP까지 포함한 4개 파일을 먼저 확인합니다.
- 학습 전 오류로 멈춘 v1 출력은 보존하고 v2 출력 폴더를 사용합니다. 이 링크로 시작한 이후에는 같은 링크로 재개하세요.
- uniform 음성표본의 재추출 순서·난수 상태는 유지하고 충돌한 표본만 재검사합니다. M2 양성 구매쌍의 고정 위치를 CPU에서 한 번 검증해 배치별 검색을 줄입니다. 학습표현은 매 배치 다시 계산합니다.
- 진행 표시용 Drive 갱신은 300초 간격이며 checkpoint는 매epoch 저장합니다. 수치 정밀도·배치·모형·손실·학습률·300epoch·최종평가 조건은 동일합니다.
- CPU 합성자료 8모형에서 기존 구현과 학습이력·모델·Adam·난수 상태가 정확히 일치했습니다. 실제 GPU 총시간 단축률은 아직 미측정입니다.

## 학교 Drive 10GB 저장 정책
- 입력 거래·상품은 검증 후 Colab 임시 디스크 `/content/hm_graphsage_inputs`로 바이트 그대로 복사해 읽습니다. 런타임 초기화 후 다시 복사되며 원본은 Drive에 보존합니다.
- 학습 중 모형은 매epoch 모델·Adam·난수 전체를 저장합니다. 300epoch 최종 평가 결과를 저장한 완료 모형은 최종 가중치·학습이력·난수만 보관하고 Adam 상태를 제외합니다. 완료 모형은 재학습하지 않습니다.
- 최종 가중치 복사 크기·해시를 확인한 뒤 기존 체크포인트를 원자적으로 교체합니다. 저장 실패 시 이전 파일을 유지하며 중단합니다.
- 입력 외 체크포인트 예상 점유는 완료 시 약3GB, 교체용 임시 파일을 포함한 학습 중 최고 약5GB입니다. 실제 파일 크기는 진행 기록에 남깁니다. 다른 실험이 같은 학교 Drive 공간을 함께 사용한다는 점을 고려하세요.

## 저장·재개·시간 확인
- **매epoch** 모델·Adam·numpy/torch/CUDA 난수를 로컬 직렬화 후 Drive에 동기 저장합니다. 이전 checkpoint는 새 파일 복사가 완료된 뒤 교체합니다.
- 끊기면 **같은 링크·코드·참조/출력 경로로 위에서부터 재실행**하세요. 완료모형은 재학습·재평가하지 않고 미완료모형은 마지막 저장 epoch 다음부터 재개합니다. 중간 배치에서 끊기면 해당 미완료 epoch는 다시 실행합니다. Colab 자동 재접속은 제공하지 않습니다.
- 손상 checkpoint/Drive 저장 오류는 파일을 보존하고 중단합니다. 원인을 확인하기 전에 결과 폴더를 삭제하지 마세요.
- 정적 입력은 한 번 준비하되 학습 임베딩은 매 배치 다시 계산합니다. uniform 음성의 동일 표본/RNG를 유지하는 이진검색 검사와 기존 저장 최적화를 사용합니다.
- 첫3epoch의 첫5배치 입력/계산 시간, 매epoch 전체 학습/Drive 저장 시간을 기록합니다. 실제 H&M/L4 전체 실행시간은 아직 미측정입니다.
- 결과 ZIP: 전체142지표, CLV 세그먼트/노출, 모든 모형간 비교, 진단, 학습이력, 시간, 설정. 미완료 ZIP은 partial로 구분합니다.
- **1시드 결과로 표준편차·95%CI·유의성을 주장하지 않습니다.** 약10시드 최종평균·CLV 배정귀속·새 독립확증을 대체하지 않습니다.

근거: [GraphSAGE 원논문](https://papers.nips.cc/paper/6703-inductive-representation-learning-on-large-graphs.pdf)의 평균 집계·자기/이웃 결합 및 층별 정규화 구조. ELU·층평균·ID/BPR는 프로젝트 추천용 변형으로 별도 명시합니다.
별도 PyG 설치 없이 PyTorch 희소행렬을 사용합니다. 실제 Colab 전체학습은 사용자 실행 단계이며 로컬 합성자료로 수식·개입 경로·재개를 검증했습니다.

같은 출력 폴더를 두 런타임에서 동시에 실행하지 마세요. 평가 후 결과 JSON 저장 전에 끊기면 재접속 시 최종 평가만 재계산될 수 있습니다.


학교 계정 jungunb@u.sogang.ac.kr로 Colab과 Drive를 연결하세요. L4 및 고용량 RAM을 권장합니다. H&M 전체 그래프의 실제 GPU 메모리·소요시간은 아직 측정하지 않았습니다. 초기 배치/epoch 시간으로 진행 비용을 확인하며 결과를 보고 설정을 바꾸지 않습니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), '런타임 유형을 L4 GPU로 설정하세요.'
GPU = torch.cuda.get_device_name(0)
assert 'L4' in GPU, f'현재 GPU: {GPU}. L4로 변경 후 실행하세요.'
SOURCE_COMMIT = '214e5d1a64d3a9a5a4c790323ae71dab8851c608'
REPO = Path('/content/clv-hm-graphsage-v2-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
sys.path.insert(0, str(REPO))
print('GPU:', GPU, '| 고정 코드:', SOURCE_COMMIT)

## 1. 복구된 학교 Drive 입력 검증 — 학습 없음
원본 거래·상품과 M2/M4 참조 ZIP의 SHA256을 검사합니다. 기존 5조각 ZIP을 재설치하지 않습니다. 전체 전처리 후 M2 특징·M3 그래프·M4 가중치도 기존 실험과 대조합니다.


In [ ]:
import hashlib
DATA = Path('/content/drive/MyDrive/논문/data')
EXPECTED_FILES = {
    'raw/hm/transactions_train.parquet': '0d90884fc1bc400161e6aeba238cba94183998bedfeb4ebbf4962e17d0c8489b',
    'raw/hm/articles.csv': '25f21882053517f7684ea43f524225a9481a7bb9aab9cf0abfc7fe8c4576a72b',
    'references/clv_m2_adaptive_hm_seed49_lastweek_results.zip': '6d5159a888319b8a443a6907922a6e36fe9954c6b717dc2b1a6ff547c1d52bed',
    'references/clv_hm_m4c_lastweek_seed49_results.zip': '11fbfbddd21f966357cc872d43891bd078741b83037e88ee39a5558498ce40fe',
}
for name, expected in EXPECTED_FILES.items():
    path = DATA / name
    assert path.is_file(), f'학교 Drive 입력 파일 없음: {path}'
    digest = hashlib.sha256()
    with path.open('rb') as source:
        for block in iter(lambda: source.read(8 * 1024 * 1024), b''):
            digest.update(block)
    assert digest.hexdigest() == expected, f'원본과 다른 입력: {name}. 이전 ZIP을 재설치하지 마세요.'
print('학교 Drive 원본 입력·참조 4개 SHA256 일치')

import clv_hm_graphsage_final8_seed49 as screen
assert Path(screen.__file__).resolve().parent == REPO.resolve(), '이전 모듈이 메모리에 남아 있습니다. 이 런타임만 다시 시작한 후 위에서부터 실행하세요.'
# Fixed raw bytes are read from the runtime disk, not repeatedly from Drive.
import shutil
LOCAL_INPUT = Path('/content/hm_graphsage_inputs')
LOCAL_INPUT.mkdir(exist_ok=True)
for key, name in [('tx_path', 'transactions_train.parquet'), ('item_meta_path', 'articles.csv')]:
    source = DATA / 'raw/hm' / name
    target = LOCAL_INPUT / name
    expected = EXPECTED_FILES['raw/hm/' + name]
    def matches(path):
        if not path.is_file():
            return False
        h = hashlib.sha256()
        with path.open('rb') as f:
            for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
                h.update(block)
        return h.hexdigest() == expected
    if not matches(target):
        temporary = target.with_suffix(target.suffix + '.partial')
        shutil.copyfile(source, temporary)
        assert matches(temporary), f'로컬 복사 해시 불일치: {name}'
        temporary.replace(target)
    screen.base.v3.SCHEMA['hm'][key] = str(target)
print('원본과 동일한 Colab 로컬 입력 준비 완료')
cfg = screen.configure(
    m2_reference=str(DATA / 'references/clv_m2_adaptive_hm_seed49_lastweek_results.zip'),
    m4_reference=str(DATA / 'references/clv_hm_m4c_lastweek_seed49_results.zip'),
    out_dir=str(DATA / 'results_v3_hm_graphsage_final8_seed49_school_v2'),
)
print('출력:', cfg.out_dir)
prepared = screen.prepare(cfg)
print('검증 완료:', list(screen.ORDER), '/ seed49 / 각300epoch')
print('진행·checkpoint·결과 저장:', prepared['run_dir'])


## 2. GraphSAGE 8개 모형 순차 학습
모형당 시작3개 epoch 및 이후5epoch마다 손실·전체 학습 시간·Drive 저장 시간을 표시합니다. 처음5배치가 끝나면 입력/음성추출과 학습계산의 배치당 표본 시간도 먼저 표시하므로 첫epoch가 끝나기 전에도 확인할 수 있습니다.

매epoch checkpoint를 Drive에 저장하고, 각 모형300epoch 완료 시 전체142지표를 평가·저장한 뒤 다음 모형으로 넘어갑니다. **로그 빈도와 무관하게 Drive 저장은 매epoch 실행**됩니다. 학습 중 다른 셀을 실행하거나 런타임을 바꾸지 마세요.

각 시간은 다음과 같습니다.
- 입력·음성추출: 같은 순서의 uniform 음성표본 생성과 배치 자료 준비/전송.
- 학습계산: 그래프 전파·BPR 손실·역전파·Adam 업데이트.
- 학습 전체: epoch의 입력·계산·로그/진행기록을 포함한 벽시계 시간.
- Drive 저장완료: 로컬 직렬화·복사/해시·Drive 저장/메타데이터 전체 시간.


## 체크포인트 휴지통 누적 방지
학습을 실행하기 전에 한 번 적용합니다. 최신본의 Drive 크기·MD5와 소유자·폴더를 검증한 뒤 같은 실행/같은 이름의 명시적으로 삭제된 이전 사본만 영구 삭제합니다. 진행 중 최신본+직전본, 완료 시 현재 최종본을 유지합니다. 일반 휴지통 전체를 비우지 않습니다. 확인 실패는 삭제 없이 중단하며 저장된 체크포인트에서 재개합니다. 다른 런타임에는 각각 적용해야 합니다. 전체 Colab 실통합은 아직 미검증입니다.


In [ ]:
# 학습 셀이 멈춘 상태에서, 입력 준비 완료 후 한 번 실행하세요.
# 이 실행의 검증된 오래된 체크포인트 휴지통 사본만 영구 삭제합니다.
assert 'screen' in globals() and 'prepared' in globals(), '먼저 기존 입력 준비 셀을 실행하세요.'
if 'drive_api' not in globals():
    from google.colab import auth
    auth.authenticate_user()  # 반드시 학교 계정 선택
    import google.auth
    from googleapiclient.discovery import build
    credentials, _ = google.auth.default()
    drive_api = build('drive', 'v3', credentials=credentials, cache_discovery=False)
import hashlib, importlib.util, urllib.request
from pathlib import Path
retention_url = 'https://raw.githubusercontent.com/jung-un/clv-m2-lightgcn-runner/7291bd66f4c62183380906cfca46bc04f97809fc/clv_drive_checkpoint_retention.py'
with urllib.request.urlopen(retention_url, timeout=60) as response:
    retention_code = response.read()
assert hashlib.sha256(retention_code).hexdigest() == 'f58f97fc3f970a1c53fe018d5caff69242fd66878fb1654592e5f144883e9044', '보존 도우미 해시 불일치'
retention_path = Path('/content/clv_drive_checkpoint_retention_6f20874.py')
retention_path.write_bytes(retention_code)
retention_spec = importlib.util.spec_from_file_location('clv_drive_checkpoint_retention_patch', retention_path)
retention = importlib.util.module_from_spec(retention_spec)
retention_spec.loader.exec_module(retention)
# Drive API 반영 지연을 최대 약5분 기다린 뒤 안전 중단합니다.
retention.prune.__kwdefaults__['retries'] = 30
retention.install(screen, drive_api, prepared)


In [ ]:
assert getattr(getattr(screen, 'original', screen).SafeProgressStore, '_retention_installed', False), '보존 셀을 먼저 실행하세요. 정리 기능 없이 학습하지 않습니다.'
result = screen.run(prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))

## 시간 기록 확인 — 학습·평가 없음
아래 셀은 저장된 시간 CSV만 읽습니다. 학습 셀이 끝났거나 중단된 뒤 실행하세요. 마지막10개 epoch를 보여주며 전체 기록은 결과 ZIP의 `timing/`에 포함됩니다. 입력 준비 총 시간은 `preparation_timing.json`에 있습니다. 표본 시간은 첫3개 epoch만 있으며, 나머지 빈칸은 미측정입니다.

In [ ]:
import pandas as pd
from IPython.display import display
timing_files = sorted((prepared['run_dir'] / 'progress' / 'timing').glob('*.csv'))
if timing_files:
    timing = pd.concat([pd.read_csv(p) for p in timing_files], ignore_index=True)
    columns = ['model_id', 'epoch', 'train_seconds', 'save_seconds',
               'sample_input_seconds_per_batch', 'sample_compute_seconds_per_batch',
               'checkpoint_local_seconds', 'checkpoint_drive_seconds', 'checkpoint_metadata_seconds']
    display(timing[columns].tail(10))
    print('단위: 초. 표본 계측은 추가 동기화 비용을 포함합니다. test 성능을 측정하지 않습니다.')
else:
    print('아직 완료 epoch의 시간 기록이 없습니다. 초기 배치 시간은 학습 출력에서 확인하세요.')

## 3. 저장된 완료 모형만 비교 — 학습·재평가 없음
8모형 절대지표와 **같은 GraphSAGE M1 대비** 변화율입니다. 미완료 모형은 빈칸이며0점이 아닙니다. 전체142지표·모든 모형 간 비교·개입 진단은 CSV에 보관됩니다. 단일 시드이므로 표준편차/신뢰구간을 만들지 않습니다.

세션이 끊겼다면1번 준비 셀까지 실행한 다음 이 셀로 바로 와도 됩니다. 전체 결과를 받으려면2번 학습 셀을 이어서 실행하세요.

In [ ]:
from IPython.display import display
result = screen.report(prepared)
metrics = list(screen.CORE)
display(result['absolute'].set_index('model')[metrics].T.reindex(columns=screen.ORDER))
against_m1 = result['comparison'].query("reference == 'M1'")
if not against_m1.empty:
    display(against_m1.pivot(index='metric', columns='model', values='relative_change_pct')
            .reindex(index=metrics, columns=screen.ORDER[1:]))
    print('GraphSAGE M1 대비 상대변화(%). 모든 비교·불리한 지표도 CSV에 보존합니다.')
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))
print('seed49 1시드 — 평균/표준편차·유의성·CLV 배정 귀속 판단 아님')

## 4. 결과 ZIP 다운로드
완료 모형의 원본 결과·1~300epoch 학습 이력·전체 비교·설정·입력 검증 기록을 받습니다. 8개 미완료 시 파일명에 `_partial`을 붙입니다. 재개용 대용량 checkpoint는 ZIP에 넣지 않고 **Drive에 그대로 유지**합니다.

In [ ]:
from google.colab import files
import shutil
output = screen.export_zip(prepared, '/content/hm_graphsage_final8_seed49_results.zip')
src = Path(output)
backup = prepared['run_dir'] / src.name
shutil.copy2(src, backup)
assert backup.stat().st_size == src.stat().st_size
print('Drive 보관:', backup, '| ZIP 크기(MB):', round(src.stat().st_size / 1024**2, 3))
print('다운로드가 멈추면 위 Drive 경로에서 직접 다운로드하세요. checkpoint는 Drive에 유지됩니다.')
files.download(str(src))